In [ ]:
# ===========================================================================================
# CELL 1: LIBRARY IMPORTS DAN KONFIGURASI AWAL
# ===========================================================================================
# Cell ini berisi semua import library yang diperlukan untuk project dan konfigurasi dasar
# ===========================================================================================

# Import library untuk manipulasi data
import pandas as pd              # Untuk manipulasi dataframe
import numpy as np               # Untuk operasi numerik dan array

# Import library untuk visualisasi
import matplotlib.pyplot as plt  # Untuk membuat plot dasar
import seaborn as sns            # Untuk visualisasi statistik yang lebih cantik

# Import library utility
from pathlib import Path         # Untuk mengelola path file/folder secara modern
import warnings                  # Untuk mengelola warning messages
import re                        # Untuk regular expression (text processing)

# Nonaktifkan warning agar output lebih bersih
warnings.filterwarnings('ignore')

# Import library untuk Natural Language Processing (NLP)
from sklearn.feature_extraction.text import TfidfVectorizer  # Untuk convert text ke numerical features
from sklearn.decomposition import TruncatedSVD              # Untuk dimensionality reduction

# Import library untuk Machine Learning
from sklearn.model_selection import KFold, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer                    # Untuk handle missing values
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Import LightGBM - algoritma gradient boosting yang cepat dan efisien
from lightgbm import LGBMRegressor

# Import library lainnya
from scipy import stats          # Untuk statistical tests dan distributions
import joblib                    # Untuk save/load model
from datetime import datetime    # Untuk timestamp

# ===========================================================================================
# KONFIGURASI VISUALISASI
# ===========================================================================================
# Set style untuk semua plot agar lebih konsisten dan profesional
sns.set_style('whitegrid')                    # Gunakan style whitegrid dari seaborn
plt.rcParams['figure.figsize'] = (12, 6)      # Set default ukuran figure menjadi 12x6 inches

print("✅ Semua library berhasil diimport!")
print("📊 Konfigurasi visualisasi telah diset!")
print("🚀 Siap untuk memulai analisis Song Popularity Detection!")

In [ ]:
# ===========================================================================================
# CELL 2: CLASS DEFINITION DAN INITIALIZATION
# ===========================================================================================
# Cell ini mendefinisikan class SongPopularityPredictor dan method __init__
# Class ini adalah blueprint untuk semua operasi prediksi popularitas lagu
# ===========================================================================================

class SongPopularityPredictor:
    """
    ===========================================================================================
    SIKLUS 4 ENHANCED: Model Asli + Comprehensive Visualizations
    ===========================================================================================
    
    Class ini menangani seluruh pipeline untuk memprediksi popularitas lagu:
    1. Loading dan exploratory data analysis
    2. Feature engineering (audio, artist, temporal, lyrics features)
    3. Data preprocessing dan encoding
    4. Model training dengan cross-validation
    5. Comprehensive visualizations (15 jenis plot)
    6. Detailed insights dan error analysis
    7. Submission file generation
    
    Attributes:
        data_path (Path): Path ke folder yang berisi train.csv dan test.csv
        output_path (Path): Path ke folder untuk menyimpan hasil output
        train_df (DataFrame): Data training
        test_df (DataFrame): Data testing
        features (list): List nama fitur yang digunakan untuk modeling
        models (dict): Dictionary untuk menyimpan trained models
        scalers (dict): Dictionary untuk menyimpan scalers (jika diperlukan)
        imputer (SimpleImputer): Imputer untuk handle missing values
        label_encoders (dict): Dictionary label encoders untuk categorical features
        oof_predictions (array): Out-of-fold predictions untuk analysis
        cv_scores (array): Cross-validation scores
    ===========================================================================================
    """

    def __init__(self, data_path='/content', output_path='./outputs'):
        """
        Initialize SongPopularityPredictor dengan konfigurasi path dan variabel internal.
        
        Args:
            data_path (str): Path ke folder data (default: '/content' untuk Google Colab)
            output_path (str): Path ke folder output (default: './outputs')
        
        Inisialisasi:
            - Convert path ke Path object untuk handling yang lebih robust
            - Inisialisasi semua variabel internal sebagai None atau empty
            - Buat output folder jika belum ada
        """
        # ===========================================================================================
        # PATH CONFIGURATION
        # ===========================================================================================
        self.data_path = Path(data_path)        # Path ke folder data (train.csv, test.csv)
        self.output_path = Path(output_path)    # Path untuk menyimpan output (submission, plots)
        
        # ===========================================================================================
        # DATA STORAGE
        # ===========================================================================================
        self.train_df = None                    # DataFrame untuk training data
        self.test_df = None                     # DataFrame untuk testing data
        
        # ===========================================================================================
        # FEATURE CONFIGURATION
        # ===========================================================================================
        self.features = []                      # List nama fitur yang akan digunakan untuk modeling
        
        # ===========================================================================================
        # MODEL & PREPROCESSING OBJECTS
        # ===========================================================================================
        self.models = {}                        # Dictionary untuk menyimpan trained models
        self.scalers = {}                       # Dictionary untuk menyimpan scalers (optional)
        self.imputer = None                     # SimpleImputer untuk handle missing values
        self.label_encoders = {}                # Dictionary label encoders untuk categorical features
        
        # ===========================================================================================
        # CATEGORICAL FEATURES TRACKING
        # ===========================================================================================
        self.categorical_features_raw = []      # List categorical features sebelum encoding
        self.categorical_features_encoded = []  # List categorical features setelah encoding
        
        # ===========================================================================================
        # RESULTS STORAGE
        # ===========================================================================================
        self.model_results = None               # DataFrame untuk menyimpan hasil evaluasi model
        self.oof_predictions = None             # Out-of-fold predictions untuk detailed analysis
        self.cv_scores = None                   # Cross-validation scores dari setiap fold
        
        # ===========================================================================================
        # SETUP OUTPUT DIRECTORY
        # ===========================================================================================
        # Buat output folder jika belum ada (parents=True: buat parent folders jika perlu)
        self.output_path.mkdir(parents=True, exist_ok=True)
        
        print("✅ SongPopularityPredictor berhasil diinisialisasi!")
        print(f"📂 Data path: {self.data_path}")
        print(f"📁 Output path: {self.output_path}")

print("✅ Class SongPopularityPredictor berhasil didefinisikan!")

In [ ]:
# ===========================================================================================
# CELL 3: METHOD LOAD_DATA DAN EDA (EXPLORATORY DATA ANALYSIS)
# ===========================================================================================
# Cell ini menambahkan method untuk loading data dan exploratory data analysis
# ===========================================================================================

# Tambahkan method load_data ke class
def load_data(self):
    """
    ===========================================================================================
    METHOD: LOAD DATA
    ===========================================================================================
    Load training dan testing data dari CSV files
    
    Proses:
    1. Load train.csv dan test.csv dari data_path
    2. Display informasi dasar tentang dataset (shape, target statistics)
    
    Returns:
        self: Mengembalikan instance untuk method chaining
    
    Raises:
        FileNotFoundError: Jika file train.csv atau test.csv tidak ditemukan
    ===========================================================================================
    """
    print("="*80)
    print("📂 LOADING DATA")
    print("="*80)
    
    # Load training data dengan pandas
    # engine='python' untuk handling lebih robust terhadap format CSV yang tidak standar
    self.train_df = pd.read_csv(self.data_path / 'train.csv', engine='python')
    
    # Load testing data
    self.test_df = pd.read_csv(self.data_path / 'test.csv', engine='python')

    # Display informasi dataset
    print(f"✓ Training data: {self.train_df.shape}")  # (num_rows, num_columns)
    print(f"✓ Testing data: {self.test_df.shape}")
    
    # Display statistik target variable (popularity)
    print(f"✓ Target range: [{self.train_df['popularity'].min():.0f}, {self.train_df['popularity'].max():.0f}]")
    print(f"✓ Target mean: {self.train_df['popularity'].mean():.2f}")
    print(f"✓ Target std: {self.train_df['popularity'].std():.2f}")

    return self

# Tambahkan method eda ke class
def eda(self):
    """
    ===========================================================================================
    METHOD: EXPLORATORY DATA ANALYSIS (EDA)
    ===========================================================================================
    Melakukan exploratory data analysis untuk memahami karakteristik data
    
    Proses:
    1. Tampilkan statistik deskriptif dari target variable (popularity)
    2. Check missing values di semua kolom
    3. Tampilkan distribusi genre (top 5)
    
    Output:
        - Basic statistics (count, mean, std, min, quartiles, max)
        - Missing values summary
        - Top 5 genres by frequency
    
    Returns:
        self: Mengembalikan instance untuk method chaining
    ===========================================================================================
    """
    print("\n" + "="*80)
    print("📊 EXPLORATORY DATA ANALYSIS")
    print("="*80)

    # ===========================================================================================
    # 1. BASIC STATISTICS
    # ===========================================================================================
    print("\nBasic Statistics:")
    print(self.train_df['popularity'].describe())
    # Output: count, mean, std, min, 25%, 50%, 75%, max

    # ===========================================================================================
    # 2. MISSING VALUES CHECK
    # ===========================================================================================
    print(f"\nMissing Values:")
    missing = self.train_df.isnull().sum()  # Hitung jumlah missing per kolom
    
    if missing.sum() > 0:
        # Jika ada missing values, tampilkan kolom yang punya missing
        print(missing[missing > 0])
    else:
        # Jika tidak ada missing
        print("  No missing values in main features")

    # ===========================================================================================
    # 3. GENRE DISTRIBUTION
    # ===========================================================================================
    print(f"\nTop 5 Genres:")
    print(self.train_df['track_genre'].value_counts().head())
    # Tampilkan 5 genre paling populer beserta jumlahnya

    return self

# Attach methods ke class SongPopularityPredictor
SongPopularityPredictor.load_data = load_data
SongPopularityPredictor.eda = eda

print("✅ Method load_data dan eda berhasil ditambahkan ke class!")

In [ ]:
# ===========================================================================================
# CELL 4: METHOD ENGINEER_FEATURES (FEATURE ENGINEERING)
# ===========================================================================================
# Cell ini menambahkan method untuk feature engineering - proses membuat fitur baru
# dari data yang sudah ada untuk meningkatkan performa model
# ===========================================================================================

def engineer_features(self):
    """
    ===========================================================================================
    METHOD: FEATURE ENGINEERING
    ===========================================================================================
    Membuat berbagai fitur baru dari data mentah untuk meningkatkan prediksi model
    
    Kategori Fitur yang Dibuat:
    1. ARTIST FEATURES: Encoding berdasarkan statistik artist
       - artist_avg_pop: Rata-rata popularitas dari artist tersebut
       - artist_song_count: Jumlah lagu dari artist di dataset
       
    2. AUDIO FEATURES: Kombinasi dan transformasi fitur audio
       - energy_x_dance: Interaksi antara energy dan danceability
       - duration_min: Durasi lagu dalam menit
       - key_mode: Kombinasi key dan mode
       - tempo_category: Kategori tempo (slow/moderate/fast/very_fast)
       
    3. TEMPORAL FEATURES: Fitur berdasarkan waktu rilis
       - years_since_release: Umur lagu
       - decade: Dekade rilis (1980, 1990, 2000, dst)
       - is_classic: Binary flag untuk lagu lawas (< 2000)
       - is_recent_hit: Binary flag untuk lagu baru (>= 2020)
       
    4. TRACK NAME FEATURES: Karakteristik dari nama lagu
       - track_name_length: Panjang nama lagu (karakter)
       - track_name_word_count: Jumlah kata dalam nama lagu
       
    5. INTERACTION FEATURES: Interaksi antar fitur
       - artist_x_dance: Interaksi artist popularity dengan danceability
       - artist_x_energy: Interaksi artist popularity dengan energy
    
    Returns:
        self: Mengembalikan instance untuk method chaining
    ===========================================================================================
    """
    print("\n" + "="*80)
    print("🔧 FEATURE ENGINEERING")
    print("="*80)

    # ===========================================================================================
    # 1. ARTIST FEATURES - TARGET ENCODING
    # ===========================================================================================
    print("[1/7] Artist features (target encoding)...")
    
    # Hitung rata-rata popularity untuk setiap artist dari training data
    # Ini adalah bentuk target encoding: encode categorical (artist) dengan statistik target
    artist_popularity_map = self.train_df.groupby('artists')['popularity'].mean()
    
    # Hitung jumlah lagu untuk setiap artist di training data
    artist_song_count_map = self.train_df.groupby('artists').size()

    # Fallback values untuk artist yang belum pernah muncul di training
    global_mean_pop = self.train_df['popularity'].mean()      # Mean popularity global
    global_mean_count = self.train_df['artists'].value_counts().mean()  # Mean song count global

    # Apply mapping ke training data
    self.train_df['artist_avg_pop'] = self.train_df['artists'].map(artist_popularity_map)
    # Apply mapping ke test data (untuk artist baru, akan jadi NaN dulu)
    self.test_df['artist_avg_pop'] = self.test_df['artists'].map(artist_popularity_map)
    
    # Fill NaN dengan global mean (untuk artist yang tidak ada di training)
    self.train_df['artist_avg_pop'] = self.train_df['artist_avg_pop'].fillna(global_mean_pop)
    self.test_df['artist_avg_pop'] = self.test_df['artist_avg_pop'].fillna(global_mean_pop)

    # Apply mapping untuk song count
    self.train_df['artist_song_count'] = self.train_df['artists'].map(artist_song_count_map)
    self.test_df['artist_song_count'] = self.test_df['artists'].map(artist_song_count_map)
    
    # Fill NaN dengan global mean
    self.train_df['artist_song_count'] = self.train_df['artist_song_count'].fillna(global_mean_count)
    self.test_df['artist_song_count'] = self.test_df['artist_song_count'].fillna(global_mean_count)

    print("   ✓ artist_avg_pop, artist_song_count")

    # ===========================================================================================
    # 2-5. FITUR LAINNYA (DIBUAT UNTUK TRAIN DAN TEST)
    # ===========================================================================================
    # Loop untuk train dan test agar fitur konsisten di kedua dataset
    for df, name in [(self.train_df, 'Train'), (self.test_df, 'Test')]:

        # =======================================================================================
        # 2. AUDIO FEATURES
        # =======================================================================================
        print(f"[2/7] Audio features ({name})...")
        
        # Interaksi energy x danceability (lagu energik DAN danceability tinggi)
        if all(col in df.columns for col in ['energy', 'danceability']):
            df['energy_x_dance'] = df['energy'] * df['danceability']
        
        # Convert durasi dari milliseconds ke minutes (lebih interpretable)
        if 'duration_ms' in df.columns:
            df['duration_min'] = df['duration_ms'] / 60000
        
        # Kombinasi key dan mode (misalnya: C Major = '0_1', D Minor = '2_0')
        if all(col in df.columns for col in ['key', 'mode']):
            df['key_mode'] = df['key'].astype(str) + '_' + df['mode'].astype(str)
        
        # Kategorisasi tempo menjadi 4 bins
        if 'tempo' in df.columns:
            df['tempo_category'] = pd.cut(df['tempo'],
                                          bins=[0, 90, 120, 150, 250],
                                          labels=['slow', 'moderate', 'fast', 'very_fast'])

        # =======================================================================================
        # 3. TEMPORAL FEATURES
        # =======================================================================================
        print(f"[3/7] Temporal features ({name})...")
        
        if 'release_year' in df.columns:
            # Umur lagu (2025 - release_year)
            df['years_since_release'] = 2025 - df['release_year']
            
            # Dekade (1980, 1990, 2000, dst)
            df['decade'] = (df['release_year'] // 10) * 10
            
            # Binary flag: lagu lawas (sebelum tahun 2000)
            df['is_classic'] = (df['release_year'] < 2000).astype(int)
            
            # Binary flag: lagu baru (2020 atau lebih baru)
            df['is_recent_hit'] = (df['release_year'] >= 2020).astype(int)

        # =======================================================================================
        # 4. TRACK NAME FEATURES
        # =======================================================================================
        print(f"[4/7] Track name features ({name})...")
        
        if 'track_name' in df.columns:
            # Clean track name: lowercase dan remove bagian-bagian yang tidak penting
            clean_name = df['track_name'].astype(str).str.lower()
            
            # Remove bagian dalam kurung/bracket (biasanya info tambahan)
            clean_name = clean_name.str.replace(r'[\(\[].*?[\)\]]', '', regex=True)
            
            # Remove bagian setelah "feat.", "with", dll (featuring artists)
            clean_name = clean_name.str.split(' - feat.').str[0]
            clean_name = clean_name.str.split(' - with').str[0]
            clean_name = clean_name.str.split(' - sped up').str[0]
            clean_name = clean_name.str.split(' - remastered').str[0]
            clean_name = clean_name.str.split(' - from').str[0]
            clean_name = clean_name.str.strip()

            # Panjang nama lagu (jumlah karakter)
            df['track_name_length'] = clean_name.str.len()
            
            # Jumlah kata dalam nama lagu
            df['track_name_word_count'] = clean_name.str.count(' ') + 1

        # =======================================================================================
        # 5. INTERACTION FEATURES
        # =======================================================================================
        print(f"[5/7] Interaction features ({name})...")
        
        # Interaksi antara artist popularity dan audio features
        if 'artist_avg_pop' in df.columns and 'danceability' in df.columns:
            # Artist populer dengan lagu danceability tinggi
            df['artist_x_dance'] = df['artist_avg_pop'] * df['danceability']
            
            # Artist populer dengan lagu energi tinggi
            df['artist_x_energy'] = df['artist_avg_pop'] * df['energy']

    print("✓ Feature engineering completed!")
    return self

# Attach method ke class
SongPopularityPredictor.engineer_features = engineer_features

print("✅ Method engineer_features berhasil ditambahkan ke class!")

In [ ]:
# ===========================================================================================
# CELL 5: METHOD PROCESS_LYRICS (NLP FEATURE EXTRACTION)
# ===========================================================================================
# Cell ini menambahkan method untuk memproses lyrics menggunakan teknik NLP
# ===========================================================================================

def process_lyrics(self, n_components=20):
    """
    ===========================================================================================
    METHOD: PROCESS LYRICS
    ===========================================================================================
    Mengekstrak fitur numerik dari lyrics menggunakan TF-IDF dan dimensionality reduction
    
    Alur Proses:
    1. TF-IDF Vectorization:
       - Convert text lyrics menjadi numerical representation
       - TF-IDF (Term Frequency - Inverse Document Frequency) memberikan weight
         pada kata berdasarkan seberapa penting kata tersebut
       - Kata yang sering muncul di banyak dokumen akan diberi weight rendah
       - Kata yang jarang tapi spesifik akan diberi weight tinggi
    
    2. Dimensionality Reduction (SVD):
       - Mengurangi ribuan fitur TF-IDF menjadi n_components fitur
       - SVD (Singular Value Decomposition) = teknik matrix factorization
       - Mengcapture pola/tema utama dalam lyrics
       - Mirip dengan PCA tapi untuk sparse matrix
    
    Parameters:
        n_components (int): Jumlah komponen SVD yang akan dibuat (default: 20)
    
    Output Features:
        lyrics_feature_0 hingga lyrics_feature_{n_components-1}
    
    Returns:
        self: Mengembalikan instance untuk method chaining
    ===========================================================================================
    """
    print("\n" + "="*80)
    print("📝 PROCESSING LYRICS (NLP)")
    print("="*80)

    # Check apakah kolom 'lyrics' ada
    if 'lyrics' not in self.train_df.columns:
        print("⚠ No lyrics column found, skipping NLP features")
        return self

    print(f"Extracting TF-IDF features (n_components={n_components})...")
    
    # ===========================================================================================
    # 1. HANDLE MISSING VALUES
    # ===========================================================================================
    # Fill missing lyrics dengan empty string
    self.train_df['lyrics'] = self.train_df['lyrics'].fillna('')
    self.test_df['lyrics'] = self.test_df['lyrics'].fillna('')

    # ===========================================================================================
    # 2. TF-IDF VECTORIZATION
    # ===========================================================================================
    # Inisialisasi TF-IDF Vectorizer dengan parameter:
    tfidf = TfidfVectorizer(
        max_features=500,       # Ambil maksimal 500 kata paling penting
        min_df=5,              # Kata harus muncul minimal di 5 dokumen
        max_df=0.8,            # Kata tidak boleh muncul di >80% dokumen (terlalu umum)
        ngram_range=(1, 2),    # Gunakan unigram (1 kata) dan bigram (2 kata)
        stop_words='english'   # Remove common words seperti 'the', 'is', 'and'
    )
    
    # Transform lyrics menjadi TF-IDF matrix
    # Output: sparse matrix dengan shape (n_samples, n_features)
    train_tfidf = tfidf.fit_transform(self.train_df['lyrics'])  # Fit dan transform training
    test_tfidf = tfidf.transform(self.test_df['lyrics'])        # Hanya transform test (no fit!)

    # ===========================================================================================
    # 3. DIMENSIONALITY REDUCTION DENGAN SVD
    # ===========================================================================================
    # Inisialisasi TruncatedSVD untuk mengurangi dimensi
    # Dari 500 features → n_components features (default: 20)
    svd = TruncatedSVD(
        n_components=n_components,  # Jumlah komponen yang diinginkan
        random_state=42             # Untuk reproducibility
    )
    
    # Apply SVD transformation
    train_lyrics_features = svd.fit_transform(train_tfidf)  # Fit dan transform training
    test_lyrics_features = svd.transform(test_tfidf)        # Hanya transform test
    
    # ===========================================================================================
    # 4. EVALUASI EXPLAINED VARIANCE
    # ===========================================================================================
    # Hitung berapa persen variance yang dijelaskan oleh komponen-komponen ini
    explained_variance = svd.explained_variance_ratio_.sum()
    print(f"✓ Explained variance: {explained_variance:.2%}")
    # Semakin tinggi explained variance, semakin baik komponen merepresentasikan data asli

    # ===========================================================================================
    # 5. CONVERT KE DATAFRAME DAN MERGE
    # ===========================================================================================
    # Buat nama kolom untuk fitur lyrics
    lyrics_cols = [f'lyrics_feature_{i}' for i in range(n_components)]
    
    # Convert numpy array ke DataFrame
    train_lyrics_df = pd.DataFrame(
        train_lyrics_features, 
        columns=lyrics_cols, 
        index=self.train_df.index  # Pastikan index sama dengan train_df
    )
    test_lyrics_df = pd.DataFrame(
        test_lyrics_features, 
        columns=lyrics_cols, 
        index=self.test_df.index   # Pastikan index sama dengan test_df
    )

    # Gabungkan lyrics features ke dataframe utama
    self.train_df = pd.concat([self.train_df, train_lyrics_df], axis=1)
    self.test_df = pd.concat([self.test_df, test_lyrics_df], axis=1)

    print(f"✓ Added {n_components} lyrics features")
    return self

# Attach method ke class
SongPopularityPredictor.process_lyrics = process_lyrics

print("✅ Method process_lyrics berhasil ditambahkan ke class!")

In [ ]:
# ===========================================================================================
# CELL 6: METHOD PREPARE_FEATURES (FINAL PREPROCESSING)
# ===========================================================================================
# Cell ini menambahkan method untuk mempersiapkan fitur final sebelum modeling
# Meliputi: feature selection, encoding categorical, imputation, type casting
# ===========================================================================================

def prepare_features(self):
    """
    ===========================================================================================
    METHOD: PREPARE FEATURES
    ===========================================================================================
    Mempersiapkan fitur final untuk modeling dengan proses:
    1. Feature selection (pilih numeric features yang relevan)
    2. Handle explicit column (convert boolean ke integer)
    3. Identify dan encode categorical features
    4. Imputation untuk handle missing values
    5. Type casting untuk categorical features
    
    Proses ini memastikan semua fitur dalam format yang tepat untuk model LightGBM
    
    Output:
        - self.features: List nama fitur yang akan digunakan untuk training
        - self.categorical_features_encoded: List categorical features yang sudah di-encode
        - self.label_encoders: Dictionary berisi encoder untuk setiap categorical feature
    
    Returns:
        self: Mengembalikan instance untuk method chaining
    ===========================================================================================
    """
    print("\n" + "="*80)
    print("🎯 PREPARING FEATURES FOR MODELING")
    print("="*80)

    # ===========================================================================================
    # 1. SELECT NUMERIC FEATURES
    # ===========================================================================================
    # Ambil semua kolom dengan tipe data numerik (int, float)
    numeric_features = self.train_df.select_dtypes(include=[np.number]).columns.tolist()

    # ===========================================================================================
    # 2. HANDLE EXPLICIT COLUMN
    # ===========================================================================================
    # Kolom 'explicit' biasanya boolean (True/False), convert ke integer (1/0)
    if 'explicit' in self.train_df.columns:
        self.train_df['explicit'] = self.train_df['explicit'].astype(int)
        self.test_df['explicit'] = self.test_df['explicit'].astype(int)
        
        # Pastikan 'explicit' ada di list numeric features
        if 'explicit' not in numeric_features:
            numeric_features.append('explicit')

    # ===========================================================================================
    # 3. EXCLUDE NON-FEATURE COLUMNS
    # ===========================================================================================
    # Kolom-kolom yang tidak boleh digunakan sebagai fitur:
    exclude_cols = [
        'popularity',      # Target variable (tidak boleh digunakan sebagai fitur!)
        'track_id',        # ID (tidak informatif untuk prediksi)
        'track_name',      # Text (sudah diekstrak fiturnya di engineer_features)
        'artists',         # Text (sudah diekstrak fiturnya sebagai artist_avg_pop, dll)
        'lyrics',          # Text (sudah diekstrak fiturnya di process_lyrics)
        'release_year'     # Sudah di-derive menjadi fitur lain (decade, years_since_release, dll)
    ]
    
    # Filter out excluded columns dari numeric_features
    numeric_features = [f for f in numeric_features if f not in exclude_cols]

    # ===========================================================================================
    # 4. IDENTIFY CATEGORICAL FEATURES
    # ===========================================================================================
    # List categorical features yang perlu di-encode
    categorical_features = [
        'track_genre',      # Genre lagu (rock, pop, jazz, dll)
        'key_mode',         # Kombinasi key + mode (dibuat di engineer_features)
        'tempo_category',   # Kategori tempo (slow/moderate/fast/very_fast)
        'decade'            # Dekade rilis (1980, 1990, 2000, dll)
    ]
    
    # Filter hanya categorical features yang benar-benar ada di dataframe
    self.categorical_features_raw = [f for f in categorical_features if f in self.train_df.columns]

    # ===========================================================================================
    # 5. ENCODE CATEGORICAL FEATURES
    # ===========================================================================================
    print("Encoding categorical features...")
    encoded_cat_features = []

    for col in self.categorical_features_raw:
        # Inisialisasi LabelEncoder untuk feature ini
        le = LabelEncoder()
        
        # Combine train dan test data untuk fitting
        # Ini memastikan encoder mengenal semua possible values dari kedua dataset
        combined_series = pd.concat([
            self.train_df[col].astype(str),
            self.test_df[col].astype(str)
        ])
        
        # Fit encoder dengan combined data
        le.fit(combined_series)

        # Transform training data (convert categorical → integer)
        self.train_df[col + '_encoded'] = le.transform(self.train_df[col].astype(str))
        
        # Transform test data
        self.test_df[col + '_encoded'] = le.transform(self.test_df[col].astype(str))

        # Simpan encoder untuk possible future use
        self.label_encoders[col] = le
        
        # Tambahkan nama kolom encoded ke list
        encoded_cat_features.append(col + '_encoded')

    # Simpan list categorical features yang sudah encoded
    self.categorical_features_encoded = encoded_cat_features
    
    # ===========================================================================================
    # 6. COMBINE ALL FEATURES
    # ===========================================================================================
    # Gabungkan numeric features dan encoded categorical features
    self.features = numeric_features + encoded_cat_features

    print(f"✓ Total features for modeling: {len(self.features)}")

    # ===========================================================================================
    # 7. IMPUTATION (HANDLE MISSING VALUES)
    # ===========================================================================================
    print("Applying imputation (median)...")
    
    # Inisialisasi SimpleImputer dengan strategy median
    # Median lebih robust terhadap outliers dibanding mean
    self.imputer = SimpleImputer(strategy='median')
    
    # Fit dan transform training data
    self.train_df[self.features] = self.imputer.fit_transform(self.train_df[self.features])
    
    # Transform test data (hanya transform, tidak fit!)
    self.test_df[self.features] = self.imputer.transform(self.test_df[self.features])

    # ===========================================================================================
    # 8. TYPE CASTING UNTUK CATEGORICAL FEATURES
    # ===========================================================================================
    # LightGBM bisa leverage categorical features jika tipe datanya 'category'
    # Ini memberikan handling khusus yang lebih optimal untuk categorical data
    for col in self.categorical_features_encoded:
        self.train_df[col] = self.train_df[col].astype('category')
        self.test_df[col] = self.test_df[col].astype('category')

    print("✓ Features prepared!")
    return self

# Attach method ke class
SongPopularityPredictor.prepare_features = prepare_features

print("✅ Method prepare_features berhasil ditambahkan ke class!")

In [ ]:
# ===========================================================================================
# CELL 7: METHOD TRAIN_MODELS (MODEL TRAINING & CROSS-VALIDATION)
# ===========================================================================================
# Cell ini menambahkan method untuk training model LightGBM dengan cross-validation
# ===========================================================================================

def train_models(self, cv_folds=5):
    """
    ===========================================================================================
    METHOD: TRAIN MODELS
    ===========================================================================================
    Melatih model LightGBM dengan K-Fold Cross-Validation
    
    Proses:
    1. Inisialisasi LightGBM Regressor dengan hyperparameters yang sudah dituning
    2. Lakukan K-Fold Cross-Validation untuk evaluasi yang robust
    3. Dapatkan Out-of-Fold (OOF) predictions untuk detailed analysis
    4. Train model final pada seluruh training data
    
    Cross-Validation:
        - Membagi data menjadi K folds (default: 5)
        - Setiap fold digunakan sebagai validation set sekali
        - Model di-train pada K-1 folds lainnya
        - Memberikan estimasi performa yang lebih reliable
    
    Out-of-Fold Predictions:
        - Setiap sample diprediksi ketika dia ada di validation fold
        - Menghasilkan predictions untuk seluruh training set
        - Digunakan untuk detailed error analysis
    
    Parameters:
        cv_folds (int): Jumlah folds untuk cross-validation (default: 5)
    
    Model Parameters:
        - n_estimators=1000: Jumlah boosting iterations
        - learning_rate=0.01: Step size untuk update model (kecil = lebih konservatif)
        - num_leaves=31: Maksimal jumlah leaves per tree (31 = default yang bagus)
        - max_depth=6: Maksimal kedalaman tree (prevent overfitting)
        - random_state=42: Untuk reproducibility
    
    Returns:
        self: Mengembalikan instance untuk method chaining
    ===========================================================================================
    """
    print("\n" + "="*80)
    print("🤖 TRAINING MODEL (LightGBM)")
    print("="*80)

    # ===========================================================================================
    # 1. PREPARE DATA
    # ===========================================================================================
    # X: Feature matrix (semua fitur yang sudah dipreparasi)
    X = self.train_df[self.features]
    
    # y: Target variable (popularity yang ingin diprediksi)
    y = self.train_df['popularity']

    # ===========================================================================================
    # 2. INITIALIZE MODEL
    # ===========================================================================================
    lgbm = LGBMRegressor(
        n_estimators=1000,      # Banyak iterasi untuk learning yang lebih baik
        learning_rate=0.01,     # Learning rate kecil = learning lebih halus
        num_leaves=31,          # Kompleksitas per tree (31 = good default)
        max_depth=6,            # Batasi kedalaman tree (prevent overfitting)
        random_state=42,        # Seed untuk reproducibility
        n_jobs=-1,              # Gunakan semua CPU cores
        verbose=-1              # Suppress output (biar tidak terlalu banyak print)
    )

    # ===========================================================================================
    # 3. CROSS-VALIDATION
    # ===========================================================================================
    print(f"Training with {cv_folds}-fold cross-validation...")
    
    # Inisialisasi KFold cross-validator
    # shuffle=True: Acak data sebelum split (untuk distribusi yang lebih merata)
    kfold = KFold(n_splits=cv_folds, shuffle=True, random_state=42)

    # Lakukan cross-validation dan dapatkan scores
    # scoring='neg_root_mean_squared_error': Metrik evaluasi (negative karena sklearn convention)
    # n_jobs=-1: Parallel processing untuk speed up
    cv_scores = cross_val_score(
        lgbm, X, y, 
        cv=kfold,
        scoring='neg_root_mean_squared_error',  # RMSE sebagai metrik
        n_jobs=-1
    )
    
    # Convert dari negative ke positive RMSE
    self.cv_scores = -cv_scores

    # ===========================================================================================
    # 4. DISPLAY CV RESULTS
    # ===========================================================================================
    print(f"\nCross-Validation Results:")
    for i, score in enumerate(self.cv_scores, 1):
        print(f"  Fold {i}: RMSE = {score:.4f}")

    # Print mean dan standard deviation
    print(f"\n✓ Mean RMSE: {self.cv_scores.mean():.4f} (+/- {self.cv_scores.std():.4f})")
    # Mean: Average performance across all folds
    # Std: Stability (lower std = more consistent)

    # ===========================================================================================
    # 5. TRAIN FINAL MODEL
    # ===========================================================================================
    print("\nTraining on full dataset...")
    # Train model pada seluruh training data untuk final model
    lgbm.fit(X, y)
    
    # Simpan model ke dictionary
    self.models['LightGBM'] = lgbm

    # ===========================================================================================
    # 6. GET OUT-OF-FOLD PREDICTIONS
    # ===========================================================================================
    print("\nGetting OOF predictions for analysis...")
    # cross_val_predict: Mirip dengan cross_val_score tapi return predictions
    # Setiap sample diprediksi ketika dia di validation fold
    self.oof_predictions = cross_val_predict(lgbm, X, y, cv=kfold, n_jobs=-1)

    # ===========================================================================================
    # 7. STORE RESULTS
    # ===========================================================================================
    # Simpan hasil CV dalam dataframe untuk easy access
    self.model_results = pd.DataFrame({
        'Model': ['LightGBM'],
        'Mean RMSE': [self.cv_scores.mean()],
        'Std RMSE': [self.cv_scores.std()]
    })

    return self

# Attach method ke class
SongPopularityPredictor.train_models = train_models

print("✅ Method train_models berhasil ditambahkan ke class!")

In [ ]:
# ===========================================================================================
# CELL 8: METHOD CREATE_COMPREHENSIVE_VISUALIZATIONS (PART 1 - Plots 1-8)
# ===========================================================================================
# Cell ini adalah bagian pertama dari method visualisasi komprehensif
# Membuat 15 plot berbeda untuk analisis mendalam performa model
# ===========================================================================================

def create_comprehensive_visualizations_part1(self):
    """
    PART 1: Setup dan Plot 1-8 dari comprehensive visualizations
    """
    print("\n" + "="*80)
    print("📊 CREATING COMPREHENSIVE VISUALIZATIONS (PART 1/2)")
    print("="*80)

    y = self.train_df['popularity']
    X = self.train_df[self.features]
    
    # Setup figure dengan 15 subplots (5 rows x 3 columns)
    self.viz_fig = plt.figure(figsize=(24, 28))
    
    # ======================================================================================
    # PLOT 1: CV SCORES BY FOLD
    # ======================================================================================
    # Bar chart menampilkan RMSE untuk setiap fold dalam cross-validation
    ax1 = plt.subplot(5, 3, 1)
    folds = [f'Fold {i+1}' for i in range(len(self.cv_scores))]
    colors = plt.cm.RdYlGn_r(np.linspace(0.3, 0.7, len(self.cv_scores)))
    bars = ax1.bar(folds, self.cv_scores, color=colors, edgecolor='black', alpha=0.8)
    ax1.axhline(y=self.cv_scores.mean(), color='red', linestyle='--', linewidth=2,
               label=f'Mean: {self.cv_scores.mean():.4f}')
    ax1.set_ylabel('RMSE', fontweight='bold')
    ax1.set_title('Cross-Validation RMSE by Fold', fontsize=14, fontweight='bold', pad=10)
    ax1.legend()
    ax1.grid(axis='y', alpha=0.3)
    # Tambahkan nilai di atas setiap bar
    for bar, score in zip(bars, self.cv_scores):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{score:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=9)
    
    # ======================================================================================
    # PLOT 2: MODEL PERFORMANCE METRICS (RMSE, MAE, R²)
    # ======================================================================================
    # Bar chart untuk 3 metrik utama: RMSE, MAE, dan R²
    ax2 = plt.subplot(5, 3, 2)
    oof_rmse = np.sqrt(mean_squared_error(y, self.oof_predictions))
    oof_mae = mean_absolute_error(y, self.oof_predictions)
    oof_r2 = r2_score(y, self.oof_predictions)
    metrics = ['RMSE', 'MAE', 'R²']
    values = [oof_rmse, oof_mae, oof_r2]
    colors_metric = ['#FF6B6B', '#4ECDC4', '#45B7D1']
    bars = ax2.bar(metrics, values, color=colors_metric, edgecolor='black', alpha=0.8)
    ax2.set_ylabel('Score', fontweight='bold')
    ax2.set_title('Model Performance Metrics', fontsize=14, fontweight='bold', pad=10)
    ax2.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, values):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{val:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    # ======================================================================================
    # PLOT 3: FEATURE IMPORTANCE (TOP 25)
    # ======================================================================================
    # Horizontal bar chart menampilkan 25 fitur paling penting
    ax3 = plt.subplot(5, 3, 3)
    feature_importance = self.models['LightGBM'].feature_importances_
    fi_df = pd.DataFrame({'Feature': self.features, 'Importance': feature_importance})
    fi_df = fi_df.nlargest(25, 'Importance')
    colors_fi = plt.cm.viridis(np.linspace(0, 1, len(fi_df)))
    ax3.barh(range(len(fi_df)), fi_df['Importance'], color=colors_fi, edgecolor='black')
    ax3.set_yticks(range(len(fi_df)))
    ax3.set_yticklabels(fi_df['Feature'], fontsize=8)
    ax3.set_title('Top 25 Feature Importance', fontsize=14, fontweight='bold', pad=10)
    ax3.set_xlabel('Importance', fontweight='bold')
    ax3.invert_yaxis()
    ax3.grid(axis='x', alpha=0.3)
    
    # ======================================================================================
    # PLOT 4: ACTUAL VS PREDICTED SCATTER PLOT
    # ======================================================================================
    # Scatter plot membandingkan nilai actual vs predicted
    # Idealnya semua titik ada di garis diagonal (perfect prediction)
    ax4 = plt.subplot(5, 3, 4)
    scatter = ax4.scatter(y, self.oof_predictions, alpha=0.4, s=10, c=y, cmap='viridis')
    ax4.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2, label='Perfect Prediction')
    ax4.set_xlabel('Actual Popularity', fontweight='bold')
    ax4.set_ylabel('Predicted Popularity', fontweight='bold')
    ax4.set_title('Actual vs Predicted (OOF)', fontsize=14, fontweight='bold', pad=10)
    ax4.legend()
    ax4.grid(alpha=0.3)
    plt.colorbar(scatter, ax=ax4, label='Actual')
    # Text box dengan metrics
    text_box = f'RMSE: {oof_rmse:.4f}\\nMAE: {oof_mae:.4f}\\nR²: {oof_r2:.4f}'
    ax4.text(0.05, 0.95, text_box, transform=ax4.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
    
    # ======================================================================================
    # PLOT 5: RESIDUALS DISTRIBUTION
    # ======================================================================================
    # Histogram dari residuals (error = actual - predicted)
    # Idealnya berbentuk normal distribution centered di 0
    ax5 = plt.subplot(5, 3, 5)
    residuals = y - self.oof_predictions
    ax5.hist(residuals, bins=60, edgecolor='black', alpha=0.7, color='coral')
    ax5.axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero Error')
    ax5.set_xlabel('Residual', fontweight='bold')
    ax5.set_ylabel('Frequency', fontweight='bold')
    ax5.set_title('Residuals Distribution', fontsize=14, fontweight='bold', pad=10)
    ax5.legend()
    ax5.grid(axis='y', alpha=0.3)
    # Statistik residuals
    skewness = stats.skew(residuals)
    kurtosis = stats.kurtosis(residuals)
    stats_text = f'Mean: {residuals.mean():.2f}\\nStd: {residuals.std():.2f}\\nSkew: {skewness:.2f}\\nKurt: {kurtosis:.2f}'
    ax5.text(0.05, 0.95, stats_text, transform=ax5.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))
    
    # ======================================================================================
    # PLOT 6: RESIDUAL PLOT
    # ======================================================================================
    # Scatter plot residuals vs predicted values
    # Idealnya tidak ada pola (random scatter around 0)
    ax6 = plt.subplot(5, 3, 6)
    scatter = ax6.scatter(self.oof_predictions, residuals, alpha=0.4, s=10, c=y, cmap='coolwarm')
    ax6.axhline(y=0, color='red', linestyle='--', linewidth=2)
    ax6.set_xlabel('Predicted Popularity', fontweight='bold')
    ax6.set_ylabel('Residual', fontweight='bold')
    ax6.set_title('Residual Plot', fontsize=14, fontweight='bold', pad=10)
    ax6.grid(alpha=0.3)
    plt.colorbar(scatter, ax=ax6, label='Actual')
    
    # ======================================================================================
    # PLOT 7: RMSE BY PREDICTION RANGE
    # ======================================================================================
    # Bar chart RMSE untuk setiap bin prediction range
    # Menunjukkan di range mana model perform baik/buruk
    ax7 = plt.subplot(5, 3, 7)
    pred_bins = pd.cut(self.oof_predictions, bins=10)
    error_by_bin = pd.DataFrame({
        'predicted': self.oof_predictions,
        'actual': y,
        'bin': pred_bins
    }).groupby('bin').apply(lambda x: np.sqrt(mean_squared_error(x['actual'], x['predicted'])))
    colors_bins = plt.cm.RdYlGn_r(np.linspace(0.3, 0.7, len(error_by_bin)))
    bars = ax7.bar(range(len(error_by_bin)), error_by_bin.values,
                  color=colors_bins, edgecolor='black', alpha=0.8)
    ax7.set_xlabel('Prediction Range', fontweight='bold')
    ax7.set_ylabel('RMSE', fontweight='bold')
    ax7.set_title('RMSE by Prediction Range', fontsize=14, fontweight='bold', pad=10)
    ax7.set_xticks(range(len(error_by_bin)))
    ax7.set_xticklabels([f'{int(interval.left)}-{int(interval.right)}'
                        for interval in error_by_bin.index], rotation=45, ha='right', fontsize=7)
    ax7.grid(axis='y', alpha=0.3)
    ax7.axhline(y=oof_rmse, color='red', linestyle='--', alpha=0.5, label=f'Overall: {oof_rmse:.2f}')
    ax7.legend()
    
    # ======================================================================================
    # PLOT 8: PREDICTION DISTRIBUTION COMPARISON
    # ======================================================================================
    # Overlay histogram actual vs predicted
    # Idealnya distribusi keduanya mirip
    ax8 = plt.subplot(5, 3, 8)
    ax8.hist(y, bins=40, alpha=0.6, label='Actual', edgecolor='black', color='blue')
    ax8.hist(self.oof_predictions, bins=40, alpha=0.6, label='Predicted', edgecolor='black', color='red')
    ax8.set_xlabel('Popularity', fontweight='bold')
    ax8.set_ylabel('Frequency', fontweight='bold')
    ax8.set_title('Actual vs Predicted Distribution', fontsize=14, fontweight='bold', pad=10)
    ax8.legend()
    ax8.grid(axis='y', alpha=0.3)
    
    # Simpan intermediate data untuk part 2
    self.viz_data = {
        'y': y,
        'X': X,
        'oof_rmse': oof_rmse,
        'oof_mae': oof_mae,
        'oof_r2': oof_r2,
        'residuals': residuals,
        'feature_importance': feature_importance,
        'fi_df': fi_df
    }
    
    print("✓ Part 1 complete (Plots 1-8)")
    return self

# Attach method ke class
SongPopularityPredictor.create_comprehensive_visualizations_part1 = create_comprehensive_visualizations_part1

print("✅ Method create_comprehensive_visualizations_part1 berhasil ditambahkan!")

In [ ]:
# ===========================================================================================
# CELL 9: METHOD CREATE_COMPREHENSIVE_VISUALIZATIONS (PART 2 - Plots 9-15 & Save)
# ===========================================================================================
# Cell ini adalah bagian kedua dari method visualisasi komprehensif
# Melanjutkan plot 9-15 dan menyimpan figure
# ===========================================================================================

def create_comprehensive_visualizations_part2(self):
    """
    PART 2: Plot 9-15 dan save figure
    """
    print("📊 CREATING COMPREHENSIVE VISUALIZATIONS (PART 2/2)")
    
    # Load data dari part 1
    y = self.viz_data['y']
    oof_rmse = self.viz_data['oof_rmse']
    residuals = self.viz_data['residuals']
    feature_importance = self.viz_data['feature_importance']
    fi_df = self.viz_data['fi_df']
    
    # ======================================================================================
    # PLOT 9: Q-Q PLOT (NORMALITY CHECK)
    # ======================================================================================
    # Q-Q plot untuk check normalitas residuals
    # Jika residuals normal, titik akan mengikuti garis diagonal
    ax9 = plt.subplot(5, 3, 9)
    stats.probplot(residuals, dist="norm", plot=ax9)
    ax9.set_title('Q-Q Plot (Normality Check)', fontsize=14, fontweight='bold', pad=10)
    ax9.grid(alpha=0.3)
    ax9.get_lines()[0].set_markerfacecolor('skyblue')
    ax9.get_lines()[0].set_markersize(5)
    ax9.get_lines()[0].set_alpha(0.6)
    
    # ======================================================================================
    # PLOT 10: ERROR BY POPULARITY RANGE
    # ======================================================================================
    # Bar chart RMSE untuk different popularity ranges
    ax10 = plt.subplot(5, 3, 10)
    ranges = [(0, 20), (20, 40), (40, 60), (60, 80), (80, 100)]
    rmse_by_range = []
    counts = []
    labels = []
    for low, high in ranges:
        mask = (y >= low) & (y < high)
        if mask.sum() > 0:
            rmse_range = np.sqrt(mean_squared_error(y[mask], self.oof_predictions[mask]))
            rmse_by_range.append(rmse_range)
            counts.append(mask.sum())
            labels.append(f'{low}-{high}')
        else:
            rmse_by_range.append(0)
            counts.append(0)
            labels.append(f'{low}-{high}')
    colors_range = plt.cm.plasma(np.linspace(0.2, 0.8, len(ranges)))
    bars = ax10.bar(labels, rmse_by_range, color=colors_range, edgecolor='black', alpha=0.8)
    ax10.set_xlabel('Popularity Range', fontweight='bold')
    ax10.set_ylabel('RMSE', fontweight='bold')
    ax10.set_title('RMSE by Popularity Range', fontsize=14, fontweight='bold', pad=10)
    ax10.grid(axis='y', alpha=0.3)
    # Tambahkan count dan RMSE di atas bar
    for bar, count, rmse in zip(bars, counts, rmse_by_range):
        if count > 0:
            height = bar.get_height()
            ax10.text(bar.get_x() + bar.get_width()/2., height,
                     f'n={count}\\n{rmse:.2f}', ha='center', va='bottom',
                     fontsize=8, fontweight='bold')
    
    # ======================================================================================
    # PLOT 11: FEATURE CATEGORIES IMPORTANCE
    # ======================================================================================
    # Horizontal bar chart total importance by feature category
    ax11 = plt.subplot(5, 3, 11)
    feature_categories = {
        'artist': [f for f in self.features if 'artist' in f],
        'genre': [f for f in self.features if 'genre' in f.lower()],
        'audio': [f for f in self.features if any(x in f for x in
                 ['energy', 'dance', 'valence', 'loud', 'tempo', 'acoustic',
                  'speech', 'instrument', 'live'])],
        'temporal': [f for f in self.features if any(x in f for x in
                    ['year', 'age', 'decade', 'recent', 'classic'])],
        'lyrics': [f for f in self.features if 'lyrics' in f],
        'track': [f for f in self.features if any(x in f for x in
                 ['track_name', 'duration', 'explicit', 'key', 'mode'])]
    }
    cat_importance = {}
    for cat, feats in feature_categories.items():
        indices = [self.features.index(f) for f in feats if f in self.features]
        if indices:
            cat_importance[cat] = feature_importance[indices].sum()
    cat_df = pd.DataFrame(list(cat_importance.items()), columns=['Category', 'Importance'])
    cat_df = cat_df.sort_values('Importance', ascending=True)
    colors_cat = plt.cm.Set3(np.linspace(0, 1, len(cat_df)))
    bars = ax11.barh(cat_df['Category'], cat_df['Importance'],
                    color=colors_cat, edgecolor='black', alpha=0.8)
    ax11.set_title('Feature Importance by Category', fontsize=14, fontweight='bold', pad=10)
    ax11.set_xlabel('Total Importance', fontweight='bold')
    ax11.grid(axis='x', alpha=0.3)
    for bar in bars:
        width = bar.get_width()
        ax11.text(width, bar.get_y() + bar.get_height()/2.,
                 f'{width:.0f}', ha='left', va='center', fontweight='bold', fontsize=9)
    
    # ======================================================================================
    # PLOT 12: WORST PREDICTIONS SCATTER
    # ======================================================================================
    # Scatter plot highlighting worst 5% predictions
    ax12 = plt.subplot(5, 3, 12)
    abs_errors = np.abs(residuals)
    worst_mask = abs_errors >= np.percentile(abs_errors, 95)
    ax12.scatter(y[~worst_mask], self.oof_predictions[~worst_mask],
                alpha=0.3, s=8, c='blue', label='Good predictions')
    ax12.scatter(y[worst_mask], self.oof_predictions[worst_mask],
                alpha=0.8, s=50, c='red', edgecolor='black', label='Top 5% errors')
    ax12.plot([y.min(), y.max()], [y.min(), y.max()], 'k--', lw=2, alpha=0.5)
    ax12.set_xlabel('Actual Popularity', fontweight='bold')
    ax12.set_ylabel('Predicted Popularity', fontweight='bold')
    ax12.set_title('Highlighting Worst Predictions', fontsize=14, fontweight='bold', pad=10)
    ax12.legend()
    ax12.grid(alpha=0.3)
    
    # ======================================================================================
    # PLOT 13: ERROR ANALYSIS BY POPULARITY BINS (BOXPLOT)
    # ======================================================================================
    # Boxplot absolute error untuk low/mid/high popularity
    ax13 = plt.subplot(5, 3, 13)
    error_analysis = pd.DataFrame({
        'actual': y,
        'predicted': self.oof_predictions,
        'abs_error': abs_errors,
        'range': pd.cut(y, bins=[0, 30, 60, 100], labels=['Low\\n(0-30)', 'Mid\\n(30-60)', 'High\\n(60-100)'])
    })
    box_data = [error_analysis[error_analysis['range'] == r]['abs_error'].values
                for r in ['Low\\n(0-30)', 'Mid\\n(30-60)', 'High\\n(60-100)']]
    bp = ax13.boxplot(box_data, labels=['Low\\n(0-30)', 'Mid\\n(30-60)', 'High\\n(60-100)'],
                     patch_artist=True)
    colors_box = ['#FF9999', '#FFD699', '#99FF99']
    for patch, color in zip(bp['boxes'], colors_box):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax13.set_ylabel('Absolute Error', fontweight='bold')
    ax13.set_title('Error Distribution by Popularity Range', fontsize=14, fontweight='bold', pad=10)
    ax13.grid(axis='y', alpha=0.3)
    
    # ======================================================================================
    # PLOT 14: TOP 10 FEATURES CONTRIBUTION
    # ======================================================================================
    # Horizontal bar dengan percentage contribution
    ax14 = plt.subplot(5, 3, 14)
    top_10_features = fi_df.head(10)
    colors_top = plt.cm.viridis(np.linspace(0, 1, len(top_10_features)))
    bars = ax14.barh(range(len(top_10_features)), top_10_features['Importance'],
                    color=colors_top, edgecolor='black')
    ax14.set_yticks(range(len(top_10_features)))
    ax14.set_yticklabels(top_10_features['Feature'], fontsize=9)
    ax14.set_title('Top 10 Most Important Features', fontsize=14, fontweight='bold', pad=10)
    ax14.set_xlabel('Importance', fontweight='bold')
    ax14.invert_yaxis()
    ax14.grid(axis='x', alpha=0.3)
    total_importance = feature_importance.sum()
    for bar, feat in zip(bars, top_10_features['Importance']):
        width = bar.get_width()
        pct = (feat / total_importance) * 100
        ax14.text(width, bar.get_y() + bar.get_height()/2.,
                 f'{pct:.1f}%', ha='left', va='center', fontweight='bold', fontsize=8)
    
    # ======================================================================================
    # PLOT 15: SUMMARY STATISTICS TABLE
    # ======================================================================================
    # Table dengan summary metrics
    ax15 = plt.subplot(5, 3, 15)
    ax15.axis('tight')
    ax15.axis('off')
    summary_stats = [
        ['Metric', 'Value'],
        ['Mean RMSE (CV)', f'{self.cv_scores.mean():.4f}'],
        ['Std RMSE (CV)', f'{self.cv_scores.std():.4f}'],
        ['OOF RMSE', f'{oof_rmse:.4f}'],
        ['OOF MAE', f'{self.viz_data["oof_mae"]:.4f}'],
        ['OOF R²', f'{self.viz_data["oof_r2"]:.4f}'],
        ['Residual Mean', f'{residuals.mean():.4f}'],
        ['Residual Std', f'{residuals.std():.4f}'],
        ['Total Features', f'{len(self.features)}'],
        ['Training Samples', f'{len(y)}']
    ]
    table = ax15.table(cellText=summary_stats, cellLoc='left', loc='center',
                      colWidths=[0.5, 0.3])
    table.auto_set_font_size(False)
    table.set_fontsize(10)
    table.scale(1, 2.5)
    for i in range(len(summary_stats)):
        if i == 0:
            table[(i, 0)].set_facecolor('#4CAF50')
            table[(i, 1)].set_facecolor('#4CAF50')
            table[(i, 0)].set_text_props(weight='bold', color='white')
            table[(i, 1)].set_text_props(weight='bold', color='white')
        else:
            table[(i, 0)].set_facecolor('#E8F5E9')
            table[(i, 1)].set_text_props(weight='bold')
    ax15.set_title('Model Summary Statistics', fontsize=14, fontweight='bold', pad=20)
    
    # ======================================================================================
    # SAVE FIGURE
    # ======================================================================================
    plt.tight_layout()
    output_file = self.output_path / 'siklus4_comprehensive_analysis.png'
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"✓ Comprehensive visualization saved: {output_file}")
    plt.show()
    
    return self

# Wrapper method yang menggabungkan part 1 dan part 2
def create_comprehensive_visualizations(self):
    """
    Main method untuk create comprehensive visualizations
    Memanggil part1 dan part2 secara berurutan
    """
    self.create_comprehensive_visualizations_part1()
    self.create_comprehensive_visualizations_part2()
    return self

# Attach methods ke class
SongPopularityPredictor.create_comprehensive_visualizations_part2 = create_comprehensive_visualizations_part2
SongPopularityPredictor.create_comprehensive_visualizations = create_comprehensive_visualizations

print("✅ Method create_comprehensive_visualizations (part2 & wrapper) berhasil ditambahkan!")

In [ ]:
# ===========================================================================================
# CELL 10: METHODS GENERATE_INSIGHTS_REPORT & ANALYZE_ERRORS
# ===========================================================================================
# Cell ini menambahkan method untuk generate detailed insights report dan error analysis
# ===========================================================================================

def generate_insights_report(self):
    """
    Generate detailed insights report dengan berbagai analisis performa model
    Mencakup: model performance, feature importance, residuals analysis, error by range,
    key insights, dan recommendations
    """
    print("\n" + "="*80)
    print("📝 DETAILED INSIGHTS REPORT")
    print("="*80)

    y = self.train_df['popularity']
    residuals = y - self.oof_predictions

    # ==================================================================================
    # 1. MODEL PERFORMANCE SUMMARY
    # ==================================================================================
    print("\n" + "─"*80)
    print("1. MODEL PERFORMANCE SUMMARY")
    print("─"*80)

    oof_rmse = np.sqrt(mean_squared_error(y, self.oof_predictions))
    oof_mae = mean_absolute_error(y, self.oof_predictions)
    oof_r2 = r2_score(y, self.oof_predictions)

    print(f"\nCross-Validation:")
    for i, score in enumerate(self.cv_scores, 1):
        print(f"  Fold {i}: RMSE = {score:.4f}")
    print(f"  Mean:    RMSE = {self.cv_scores.mean():.4f}")
    print(f"  Std:     RMSE = {self.cv_scores.std():.4f}")

    print(f"\nOut-of-Fold Performance:")
    print(f"  RMSE: {oof_rmse:.4f}")  # Lower is better
    print(f"  MAE:  {oof_mae:.4f}")   # Lower is better
    print(f"  R²:   {oof_r2:.4f}")    # Higher is better (max = 1.0)
    print(f"  MAPE: {np.mean(np.abs((y - self.oof_predictions) / (y + 1))) * 100:.2f}%")

    # ==================================================================================
    # 2. TOP 30 MOST IMPORTANT FEATURES
    # ==================================================================================
    print("\n" + "─"*80)
    print("2. TOP 30 MOST IMPORTANT FEATURES")
    print("─"*80)

    feature_importance = self.models['LightGBM'].feature_importances_
    fi_df = pd.DataFrame({
        'Feature': self.features,
        'Importance': feature_importance,
        'Importance_Pct': (feature_importance / feature_importance.sum()) * 100
    })
    fi_df = fi_df.sort_values('Importance', ascending=False).head(30)
    fi_df.index = range(1, 31)
    print(fi_df.to_string())

    # ==================================================================================
    # 3. RESIDUALS ANALYSIS
    # ==================================================================================
    print("\n" + "─"*80)
    print("3. RESIDUALS ANALYSIS")
    print("─"*80)

    print(f"\nResidual Statistics:")
    print(f"  Mean:     {residuals.mean():.4f}")     # Should be close to 0
    print(f"  Median:   {residuals.median():.4f}")   # Should be close to 0
    print(f"  Std:      {residuals.std():.4f}")
    print(f"  Min:      {residuals.min():.4f}")
    print(f"  Max:      {residuals.max():.4f}")
    print(f"  Skewness: {stats.skew(residuals):.4f}")     # Should be close to 0 for symmetry
    print(f"  Kurtosis: {stats.kurtosis(residuals):.4f}") # Excess kurtosis

    # Normality test (Shapiro-Wilk)
    shapiro_stat, shapiro_p = stats.shapiro(residuals[:5000])  # Sample untuk dataset besar
    print(f"\nShapiro-Wilk Normality Test:")
    print(f"  Statistic: {shapiro_stat:.4f}")
    print(f"  P-value:   {shapiro_p:.6f}")
    if shapiro_p > 0.05:
        print(f"  → Residuals appear normally distributed")
    else:
        print(f"  → Residuals deviate from normal distribution")

    # ==================================================================================
    # 4. ERROR ANALYSIS BY POPULARITY RANGE
    # ==================================================================================
    print("\n" + "─"*80)
    print("4. ERROR ANALYSIS BY POPULARITY RANGE")
    print("─"*80)

    ranges = [(0, 20, 'Very Low'), (20, 40, 'Low'), (40, 60, 'Medium'),
             (60, 80, 'High'), (80, 100, 'Very High')]

    print(f"{'Range':<15} {'Label':<12} {'Count':<8} {'RMSE':<10} {'MAE':<10} {'Bias':<10}")
    print("─"*80)

    for low, high, label in ranges:
        mask = (y >= low) & (y < high)
        if mask.sum() > 0:
            range_rmse = np.sqrt(mean_squared_error(y[mask], self.oof_predictions[mask]))
            range_mae = mean_absolute_error(y[mask], self.oof_predictions[mask])
            bias = (y[mask] - self.oof_predictions[mask]).mean()
            print(f"{low:>2}-{high:<3}       {label:<12} {mask.sum():<8} {range_rmse:<10.4f} {range_mae:<10.4f} {bias:<10.4f}")

    # ==================================================================================
    # 5. KEY INSIGHTS & RECOMMENDATIONS
    # ==================================================================================
    print("\n" + "─"*80)
    print("5. KEY INSIGHTS & RECOMMENDATIONS")
    print("─"*80)

    print("\n✓ Model Performance:")
    if oof_rmse < 16.0:
        print(f"  • EXCELLENT! RMSE {oof_rmse:.4f} is below 16.0 target 🎯")
    elif oof_rmse < 16.3:
        print(f"  • VERY GOOD! RMSE {oof_rmse:.4f} is competitive 👍")
    elif oof_rmse < 16.5:
        print(f"  • GOOD baseline at {oof_rmse:.4f}, some room for improvement 📈")
    else:
        print(f"  • Baseline at {oof_rmse:.4f}, consider additional features 🔧")

    print("\n✓ Bias Analysis:")
    if abs(residuals.mean()) < 0.5:
        print(f"  • Unbiased predictions (mean residual: {residuals.mean():.4f}) ✓")
    elif residuals.mean() < 0:
        print(f"  • Slight over-prediction tendency (mean: {residuals.mean():.4f})")
    else:
        print(f"  • Slight under-prediction tendency (mean: {residuals.mean():.4f})")

    print("\n✓ Error Distribution:")
    skewness = stats.skew(residuals)
    if abs(skewness) < 0.3:
        print(f"  • Symmetric error distribution (skew: {skewness:.4f}) ✓")
    elif skewness > 0:
        print(f"  • Right-skewed errors (skew: {skewness:.4f})")
        print(f"    → Model tends to under-predict high popularity songs")
    else:
        print(f"  • Left-skewed errors (skew: {skewness:.4f})")
        print(f"    → Model tends to over-predict low popularity songs")

    print("\n✓ Feature Insights:")
    top_5 = fi_df.head(5)
    print(f"  Top 5 features account for {top_5['Importance_Pct'].sum():.1f}% of importance:")
    for idx, row in top_5.iterrows():
        print(f"    {idx}. {row['Feature']}: {row['Importance_Pct']:.1f}%")

    print("\n✓ Recommendations for Next Iteration:")
    if oof_rmse > 16.2:
        print("  • Consider adding more advanced features:")
        print("    - Artist statistics (std, median, momentum)")
        print("    - Genre statistics and interactions")
        print("    - More audio feature interactions")
    if self.cv_scores.std() > 0.1:
        print(f"  • High CV variance ({self.cv_scores.std():.4f}) - consider:")
        print("    - Ensemble methods for stability")
        print("    - Two-stage modeling for different ranges")

    return self

def analyze_errors(self, n=20):
    """
    Analisis n worst predictions dengan detailed pattern analysis
    Mencari pola pada prediksi terburuk: genre, temporal, artist patterns
    """
    print("\n" + "="*80)
    print(f"🔍 TOP {n} WORST PREDICTIONS ANALYSIS")
    print("="*80)

    y = self.train_df['popularity']
    analysis_df = self.train_df.copy()
    analysis_df['oof_prediction'] = self.oof_predictions
    analysis_df['residual'] = y - self.oof_predictions
    analysis_df['abs_error'] = np.abs(analysis_df['residual'])

    # Ambil n worst predictions
    worst_errors = analysis_df.nlargest(n, 'abs_error')

    display_cols = ['track_name', 'artists', 'track_genre', 'release_year',
                   'popularity', 'oof_prediction', 'abs_error', 'artist_avg_pop']
    display_cols = [c for c in display_cols if c in worst_errors.columns]

    print("\nTop Worst Predictions:")
    print(worst_errors[display_cols].to_string(index=False))

    # ===================================================================================
    # PATTERN ANALYSIS
    # ===================================================================================
    print("\n" + "─"*80)
    print("PATTERN ANALYSIS")
    print("─"*80)

    # Genre distribution
    print("\n1. Genre Distribution in Worst Errors:")
    if 'track_genre' in worst_errors.columns:
        genre_counts = worst_errors['track_genre'].value_counts().head(5)
        total_genres = self.train_df['track_genre'].value_counts()
        print(f"\n{'Genre':<20} {'Errors':<10} {'Total':<10} {'Error Rate':<15}")
        print("─"*60)
        for genre, count in genre_counts.items():
            total = total_genres.get(genre, 0)
            rate = (count / total * 100) if total > 0 else 0
            print(f"{genre:<20} {count:<10} {total:<10} {rate:.2f}%")

    # Temporal patterns
    print("\n2. Temporal Patterns in Errors:")
    if 'release_year' in worst_errors.columns:
        year_groups = worst_errors.groupby(pd.cut(worst_errors['release_year'],
                                                   bins=[0, 1990, 2000, 2010, 2020, 2025])).size()
        print("\nErrors by decade:")
        for decade, count in year_groups.items():
            if count > 0:
                print(f"  {decade}: {count} songs")

    # Error direction
    print("\n3. Error Direction Analysis:")
    over_pred = (worst_errors['residual'] < 0).sum()
    under_pred = (worst_errors['residual'] > 0).sum()
    print(f"  Over-predictions (model too high):  {over_pred} ({over_pred/n*100:.1f}%)")
    print(f"  Under-predictions (model too low):  {under_pred} ({under_pred/n*100:.1f}%)")

    if over_pred > under_pred:
        print("\n  → Model tends to over-predict unpopular songs")
    elif under_pred > over_pred:
        print("\n  → Model tends to under-predict popular songs")

    # Artist popularity vs actual
    print("\n4. Artist Popularity vs Actual Popularity:")
    if 'artist_avg_pop' in worst_errors.columns:
        high_artist_low_song = ((worst_errors['artist_avg_pop'] > 60) &
                               (worst_errors['popularity'] < 40)).sum()
        low_artist_high_song = ((worst_errors['artist_avg_pop'] < 40) &
                               (worst_errors['popularity'] > 60)).sum()

        print(f"  High-artist/Low-song mismatches: {high_artist_low_song}")
        print(f"  Low-artist/High-song mismatches: {low_artist_high_song}")

    print("\n5. Recommendations:")
    print("  • Investigate genres with high error rates")
    print("  • Consider genre-specific models or features")
    print("  • Add features to detect outlier songs")
    print("  • Consider temporal trends patterns")

    return worst_errors

# Attach methods ke class
SongPopularityPredictor.generate_insights_report = generate_insights_report
SongPopularityPredictor.analyze_errors = analyze_errors

print("✅ Methods generate_insights_report dan analyze_errors berhasil ditambahkan!")

In [ ]:
# ===========================================================================================
# CELL 11: METHODS CREATE_SUBMISSION & RUN_FULL_PIPELINE
# ===========================================================================================
# Cell ini menambahkan method untuk create submission file dan run full pipeline
# ===========================================================================================

def create_submission(self, predictions, filename='submission.csv'):
    """
    ===========================================================================================
    METHOD: CREATE SUBMISSION
    ===========================================================================================
    Membuat file submission untuk kompetisi/testing
    
    Proses:
    1. Buat DataFrame dengan track_id dan predictions
    2. Clip predictions ke range [0, 100] (nilai popularity harus dalam range ini)
    3. Save ke CSV file
    4. Display statistik submission
    
    Parameters:
        predictions (array): Array predictions untuk test data
        filename (str): Nama file output (default: 'submission.csv')
    
    Returns:
        submission (DataFrame): DataFrame berisi track_id dan popularity predictions
    ===========================================================================================
    """
    print("\n" + "="*80)
    print(f"📤 CREATING SUBMISSION: {filename}")
    print("="*80)

    # Buat DataFrame submission dengan format: track_id, popularity
    submission = pd.DataFrame({
        'track_id': self.test_df['track_id'],      # ID dari test set
        'popularity': predictions                   # Predictions dari model
    })

    # Clip predictions ke range [0, 100]
    # Beberapa predictions mungkin < 0 atau > 100, tapi popularity harus dalam range ini
    submission['popularity'] = np.clip(submission['popularity'], 0, 100)

    # Save ke CSV
    output_path = self.output_path / filename
    submission.to_csv(output_path, index=False)

    # Display info
    print(f"✓ Saved: {output_path}")
    print(f"  • Predictions: {len(submission)}")
    print(f"  • Range: [{submission['popularity'].min():.2f}, {submission['popularity'].max():.2f}]")
    print(f"  • Mean:  {submission['popularity'].mean():.2f}")
    print(f"  • Std:   {submission['popularity'].std():.2f}")

    return submission

def run_full_pipeline(self):
    """
    ===========================================================================================
    METHOD: RUN FULL PIPELINE
    ===========================================================================================
    Menjalankan seluruh pipeline dari awal hingga akhir
    
    Pipeline Steps:
    1. Load data (train.csv, test.csv)
    2. Exploratory Data Analysis (EDA)
    3. Feature Engineering (artist, audio, temporal, track name, interaction features)
    4. Process Lyrics (TF-IDF + SVD for NLP features)
    5. Prepare Features (encoding, imputation, type casting)
    6. Train Models (LightGBM with 5-fold CV)
    7. Create Comprehensive Visualizations (15 plots)
    8. Generate Insights Report (detailed analysis)
    9. Analyze Errors (worst predictions analysis)
    10. Create Submission File
    
    Output Files:
        - submission_siklus4_enhanced.csv: Predictions untuk test set
        - siklus4_comprehensive_analysis.png: 15 visualizations
    
    Returns:
        self: Mengembalikan instance dengan semua results
    ===========================================================================================
    """
    print("\n" + "🎵"*40)
    print("SIKLUS 4 ENHANCED")
    print("Model Asli + Comprehensive Visualizations + Detailed Insights")
    print("🎵"*40)

    # =======================================================================================
    # PIPELINE EXECUTION
    # =======================================================================================
    
    # Step 1-2: Load dan EDA
    self.load_data()          # Load train.csv dan test.csv
    self.eda()                # Exploratory data analysis
    
    # Step 3-5: Feature Engineering dan Preparation
    self.engineer_features()  # Buat berbagai fitur baru
    self.process_lyrics(n_components=20)  # Extract lyrics features dengan NLP
    self.prepare_features()   # Final preparation: encoding, imputation, type casting
    
    # Step 6: Model Training
    self.train_models(cv_folds=5)  # Train LightGBM dengan 5-fold CV

    # Step 7-9: Analysis dan Visualizations
    self.create_comprehensive_visualizations()  # 15 comprehensive plots
    self.generate_insights_report()             # Detailed insights dan recommendations
    self.analyze_errors(n=20)                   # Analisis 20 worst predictions

    # Step 10: Create Submission
    # Prediksi untuk test set menggunakan trained model
    X_test = self.test_df[self.features]
    predictions = self.models['LightGBM'].predict(X_test)
    self.create_submission(predictions, 'submission_siklus4_enhanced.csv')

    # =======================================================================================
    # FINAL SUMMARY
    # =======================================================================================
    print("\n" + "="*80)
    print("✅ SIKLUS 4 ENHANCED COMPLETE!")
    print("="*80)

    y = self.train_df['popularity']
    oof_rmse = np.sqrt(mean_squared_error(y, self.oof_predictions))

    print(f"\n🎯 FINAL OOF RMSE: {oof_rmse:.4f}")
    print(f"📊 Total Features: {len(self.features)}")
    print(f"🔄 CV Folds: 5")
    print(f"📈 CV Mean RMSE: {self.cv_scores.mean():.4f} ± {self.cv_scores.std():.4f}")

    # Performance evaluation
    if oof_rmse < 16.10:
        print(f"\n🏆 EXCELLENT! Competitive performance!")
    elif oof_rmse < 16.30:
        print(f"\n🎉 VERY GOOD! Strong baseline!")
    else:
        print(f"\n💪 GOOD! Ready for next iteration!")

    # Output files info
    print("\n📁 Output Files:")
    print(f"  • submission_siklus4_enhanced.csv")
    print(f"  • siklus4_comprehensive_analysis.png (15 visualizations)")

    # Next steps
    print("\n📊 Next Steps:")
    print("  1. Review comprehensive visualizations for insights")
    print("  2. Check error patterns in worst predictions")
    print("  3. Consider improvements based on insights report")
    print("  4. Iterate with targeted feature engineering")

    return self

# Attach methods ke class
SongPopularityPredictor.create_submission = create_submission
SongPopularityPredictor.run_full_pipeline = run_full_pipeline

print("✅ Methods create_submission dan run_full_pipeline berhasil ditambahkan!")

In [ ]:
# ===========================================================================================
# CELL 12: MAIN EXECUTION
# ===========================================================================================
# Cell ini menjalankan seluruh pipeline untuk Song Popularity Detection
# 
# CARA PENGGUNAAN:
# 1. Pastikan semua cell sebelumnya sudah dijalankan (Cell 1-11)
# 2. Sesuaikan data_path jika diperlukan:
#    - Untuk Google Colab: data_path='/content'
#    - Untuk local: data_path='./data' atau path lain sesuai lokasi data
# 3. Sesuaikan output_path jika diperlukan (default: './outputs')
# 4. Run cell ini untuk menjalankan full pipeline
#
# OUTPUT:
# - Console output: Progress dan hasil dari setiap step
# - File: submission_siklus4_enhanced.csv (predictions untuk test set)
# - File: siklus4_comprehensive_analysis.png (15 visualizations)
# ===========================================================================================

if __name__ == "__main__":
    # =======================================================================================
    # KONFIGURASI PATH
    # =======================================================================================
    # Sesuaikan path ini dengan lokasi data Anda
    # Untuk Google Colab, gunakan '/content'
    # Untuk local, gunakan path ke folder yang berisi train.csv dan test.csv
    
    DATA_PATH = '/content'        # Path ke folder data
    OUTPUT_PATH = './outputs'     # Path untuk menyimpan output
    
    # =======================================================================================
    # INISIALISASI PREDICTOR
    # =======================================================================================
    # Buat instance dari SongPopularityPredictor
    predictor = SongPopularityPredictor(
        data_path=DATA_PATH, 
        output_path=OUTPUT_PATH
    )
    
    # =======================================================================================
    # RUN FULL PIPELINE
    # =======================================================================================
    # Jalankan seluruh pipeline:
    # 1. Load data
    # 2. EDA
    # 3. Feature engineering
    # 4. Process lyrics (NLP)
    # 5. Prepare features
    # 6. Train models
    # 7. Create visualizations
    # 8. Generate insights
    # 9. Analyze errors
    # 10. Create submission
    
    predictor.run_full_pipeline()
    
    # =======================================================================================
    # AKSES RESULTS
    # =======================================================================================
    # Setelah pipeline selesai, Anda bisa mengakses berbagai results:
    
    # Model yang sudah trained
    # model = predictor.models['LightGBM']
    
    # Cross-validation scores
    # cv_scores = predictor.cv_scores
    
    # Out-of-fold predictions
    # oof_preds = predictor.oof_predictions
    
    # Feature list yang digunakan
    # features = predictor.features
    
    # Training dan test dataframes (sudah dengan engineered features)
    # train_df = predictor.train_df
    # test_df = predictor.test_df
    
    # =======================================================================================
    # CUSTOM USAGE (OPTIONAL)
    # =======================================================================================
    # Jika Anda ingin menjalankan step-by-step secara manual (tidak menggunakan run_full_pipeline),
    # uncomment dan gunakan code berikut:
    
    # predictor.load_data()
    # predictor.eda()
    # predictor.engineer_features()
    # predictor.process_lyrics(n_components=20)
    # predictor.prepare_features()
    # predictor.train_models(cv_folds=5)
    # predictor.create_comprehensive_visualizations()
    # predictor.generate_insights_report()
    # predictor.analyze_errors(n=20)
    
    # # Make predictions
    # X_test = predictor.test_df[predictor.features]
    # predictions = predictor.models['LightGBM'].predict(X_test)
    # predictor.create_submission(predictions, 'submission_custom.csv')
    
    print("\n" + "="*80)
    print("🎉 SEMUA PROSES SELESAI!")
    print("="*80)
    print("\n📂 Check output folder untuk hasil:")
    print("   • submission_siklus4_enhanced.csv")
    print("   • siklus4_comprehensive_analysis.png")
    print("\n✨ Happy predicting! ✨")